# 07 · Concept LM Training
End-to-end training loop with joint NTP + boundary load balancing losses.
Demonstrates decoupled muP optimizer, loss tracking, checkpoint saving.

**Papers:** DLCM ([2512.24617](https://arxiv.org/abs/2512.24617)), CoCoMix ([2502.08524](https://arxiv.org/abs/2502.08524)), COCONUT ([2412.06769](https://arxiv.org/abs/2412.06769))

In [ ]:
import sys; sys.path.insert(0, '..')
import torch, matplotlib.pyplot as plt
from src.model import ConceptLM, ConceptLMConfig
from src.train import build_optimizer_decoupled_mup, TrainConfig

## 1. Instantiate model and inspect parameter groups

In [ ]:
cfg = ConceptLMConfig(
    d_token=128, n_token_layers=2, n_token_heads=4,
    d_concept=256, n_concept_layers=4, n_concept_heads=4,
    d_scan=64, vocab_size=50257, max_seq_len=512,
    target_ratio=4, aux_loss_weight=0.01,
)
model = ConceptLM(cfg)
total = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total:,}")
print()
for name, module in model.named_children():
    n_p = sum(p.numel() for p in module.parameters())
    print(f"  {name:25s}: {n_p:>10,} params")

## 2. Decoupled muP optimizer

In [ ]:
train_cfg = TrainConfig(lr_token=3e-4, lr_concept=1.5e-4, lr_boundary=3e-4, weight_decay=0.1)
optimizer = build_optimizer_decoupled_mup(model, train_cfg)

print("Optimizer parameter groups:")
for g in optimizer.param_groups:
    n = sum(p.numel() for p in g['params'])
    print(f"  [{g['name']:15s}] lr={g['lr']:.2e}  params={n:,}")

## 3. Single forward pass with loss

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device).train()

torch.manual_seed(42)
B, L = 2, 64
input_ids = torch.randint(0, cfg.vocab_size, (B, L), device=device)
labels    = input_ids.clone()

logits, loss, aux_loss = model(input_ids, labels=labels)
print(f"logits shape: {logits.shape}")
print(f"NTP loss:     {loss.item():.4f}")
print(f"Aux loss:     {aux_loss.item():.6f}")

## 4. Mini training loop (smoke test — 50 steps)

In [ ]:
losses, aux_losses = [], []
model.train(); optimizer.zero_grad()

for step in range(50):
    ids  = torch.randint(0, cfg.vocab_size, (2, 48), device=device)
    _, loss_s, aux_s = model(ids, labels=ids)
    loss_s.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step(); optimizer.zero_grad()
    losses.append(loss_s.item())
    aux_losses.append(aux_s.item())
    if step % 10 == 0:
        print(f"step {step:3d} | loss={loss_s.item():.4f} | aux={aux_s.item():.6f}")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3))
a1.plot(losses, color='#2E75B6'); a1.set_title('NTP Loss'); a1.set_xlabel('Step')
a2.plot(aux_losses, color='#C00000'); a2.set_title('Boundary Aux Loss'); a2.set_xlabel('Step')
plt.tight_layout(); plt.show()

## 5. Boundary rate monitoring

In [ ]:
model.eval()
with torch.no_grad():
    ids_eval = torch.randint(0, cfg.vocab_size, (4, 64), device=device)
    H = model.encoder(ids_eval)
    b_eval, _ = model.boundary(H, training=False)
    n_concepts = b_eval.sum(dim=1)
    actual_ratios = 64 / n_concepts.float()
    print(f"Target R={cfg.target_ratio}")
    print(f"Actual tokens/concept per sequence: {actual_ratios.tolist()}")
    print(f"Mean: {actual_ratios.mean():.2f}  (expect ~{cfg.target_ratio} after full training)")